# SChain Distributed Expand with Neon PostgreSQL

Run this same notebook in one or more independent Colab runtimes. Every runtime that uses the same `EXPERIMENT_ID` joins the same durable experiment, automatically claims different company tasks, and resumes from PostgreSQL after restarts. Google Drive and session lock files are not used.

In [ ]:
# User settings. Use identical compatibility settings in every worker.
EXPERIMENT_ID = "bmw-supply-chain-v1"
STATE_BACKEND = "postgres"
DISTRIBUTED_MODE = True
SAVE_TO_NEO4J = True
RUN_MODE = "AUTO"
MAX_TIER = 2
TASKS_THIS_SESSION = 10
VLLM_MAX_MODEL_LEN = 8192
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
assert STATE_BACKEND == "postgres" and DISTRIBUTED_MODE

## Install or update SChain

A new runtime clones `mirza`. An existing runtime fast-forwards it and refuses to overwrite local changes. The exact checked-out commit is printed and recorded with the worker.

In [ ]:
import os, pathlib, subprocess
REPO_DIR = pathlib.Path('/content/Schain')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'mirza', '--single-branch', 'https://github.com/ErfanBabakhani/Schain.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'diff', '--quiet'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'mirza'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', 'mirza'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'mirza'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', REPO_DIR)
print('Branch: mirza')
print('Commit:', COMMIT)
os.chdir(REPO_DIR)

In [ ]:
!python -m pip install -q -e ".[postgres]"
!python -m pip install -q vllm

## Load secrets and verify services

Create `NEON_DATABASE_URL` in Colab Secrets. When Save is enabled, also create `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD`. Secret values and connection strings are never printed.

In [ ]:
from google.colab import userdata
required = ['NEON_DATABASE_URL'] + (['NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD'] if SAVE_TO_NEO4J else [])
for name in required:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f'Missing Colab Secret: {name}')
    os.environ[name] = value
print('Required secrets loaded without displaying their values.')

In [ ]:
import psycopg
with psycopg.connect(os.environ['NEON_DATABASE_URL'], connect_timeout=20) as connection:
    connection.execute('SELECT 1').fetchone()
print('Neon PostgreSQL connectivity verified.')
if SAVE_TO_NEO4J:
    from neo4j import GraphDatabase
    probe = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD']))
    try:
        probe.verify_connectivity()
        print('Neo4j connectivity verified.')
    finally:
        probe.close()
else:
    print('Save is disabled; Neo4j was not contacted.')

## Start the local Qwen vLLM server

Each Colab owns its own model server. The cell reuses a healthy server when run again and writes logs to a unique local session file. Those logs are disposable; authoritative progress is in Neon.

In [ ]:
import httpx, time, uuid
VLLM_URL = 'http://127.0.0.1:8000/v1'
def vllm_ready():
    try:
        return httpx.get(VLLM_URL + '/models', timeout=2).status_code == 200
    except Exception:
        return False
if not vllm_ready():
    log_path = pathlib.Path('/content') / f'vllm-{uuid.uuid4().hex[:10]}.log'
    log_handle = log_path.open('w')
    vllm_process = subprocess.Popen([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL, '--host', '127.0.0.1', '--port', '8000',
        '--max-model-len', str(VLLM_MAX_MODEL_LEN),
        '--gpu-memory-utilization', '0.88', '--dtype', 'auto'
    ], stdout=log_handle, stderr=subprocess.STDOUT)
    deadline = time.time() + 600
    while time.time() < deadline and not vllm_ready():
        if vllm_process.poll() is not None:
            raise RuntimeError(f'vLLM exited; inspect {log_path}')
        time.sleep(5)
    if not vllm_ready():
        raise TimeoutError(f'vLLM did not become ready; inspect {log_path}')
    print('vLLM ready. Local log:', log_path.name)
else:
    print('Existing vLLM server is ready.')

## Build the compatible experiment configuration

The first worker creates the experiment and seeds. Later workers join only when their ontology, prompts, model, validation, seeds, maximum tier, and Save policy match the stored manifest. A completed experiment is reported, not silently restarted.

In [ ]:
from schain.config import load_config
config = load_config('config/expansion.yaml')
config['state'] = {'backend': 'postgres', 'experiment_id': EXPERIMENT_ID, 'database_url_env': 'NEON_DATABASE_URL'}
config['distributed'] = {
    'enabled': True, 'worker_concurrency': 1, 'task_lease_seconds': 600,
    'heartbeat_seconds': 30, 'max_task_attempts': 3, 'poll_seconds': 3.0,
    'connection_pool_min': 1, 'connection_pool_max': 4, 'event_retention_days': 30
}
config.setdefault('expansion', {})['max_tier'] = MAX_TIER
config.setdefault('llm', {}).update({'model': MODEL, 'base_url': VLLM_URL})
config['save'] = {'enabled': SAVE_TO_NEO4J, 'backend': 'neo4j', 'mode': 'incremental', 'max_attempts': 3, 'retry_backoff_seconds': 1.0}
config.setdefault('checkpoint_store', {})['enabled'] = False
print('Configuration ready for experiment:', EXPERIMENT_ID)

## Run one worker session (safe to run again)

Run this cell in every Colab. Each execution registers a new worker session, claims up to `TASKS_THIS_SESSION` distinct tasks, checkpoints to Neon, drains Neo4j outbox work when enabled, and stops cleanly. Running it again continues the same experiment.

In [ ]:
import json
from schain.distributed.runner import run_distributed_expand
session_result = run_distributed_expand(config, tasks_this_session=TASKS_THIS_SESSION)
print(json.dumps(session_result, indent=2, default=str))

## Show shared progress and export Graph/Audit JSON

These exports are snapshots derived from PostgreSQL. They are unique to the experiment and do not replace the authoritative database state.

In [ ]:
from schain.distributed.postgres import PostgresRepository
from schain.distributed.export import export_distributed_artifacts
settings = config['distributed']
repo = PostgresRepository(os.environ['NEON_DATABASE_URL'], pool_min=1, pool_max=2)
try:
    print(json.dumps(repo.progress(EXPERIMENT_ID), indent=2, default=str))
    exports = export_distributed_artifacts(repo, EXPERIMENT_ID, '/content/schain_exports')
    print(json.dumps(exports, indent=2))
finally:
    repo.close()

### Lifecycle meanings

- **Created**: this worker atomically created a new experiment and initialized seeds once.
- **Joined**: the experiment already existed and compatibility checks passed.
- **Worker session finished**: only this runtime's configured task allowance ended; other work may remain.
- **Experiment completed**: no pending, retrying, or leased tasks and no required Neo4j outbox work remain.
- **Runtime crash**: another worker reclaims its task after lease expiry; stale results are rejected by the fencing token.